# Rowan Adapter Interactive Test

This notebook does not train and does not push to GitHub. It clones the API repo, finds your trained Rowan adapters in Google Drive, copies them into the repo, and starts an interactive level that prints Rowan's reply and reward score.


## 1. Mount Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Clone Repo And Install Runtime Dependencies


In [ ]:
from pathlib import Path
import os
import shutil

REPO_URL = 'https://github.com/pianomaster99/isekai.git'
REPO_DIR = Path('/content/isekai')

shutil.rmtree(REPO_DIR, ignore_errors=True)
!git clone $REPO_URL $REPO_DIR
%cd /content/isekai
!pip install -U transformers peft accelerate safetensors
!pip uninstall -y torchao


## 3. Find And Copy Trained Adapters From Drive


In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive')
PREFERRED_DRIVE_MODEL_DIR = DRIVE_ROOT / 'isekai-rowan-models'
ADAPTER_NAMES = [
    'rowan-qwen3-1.7b-sft',
    'rowan-qwen3-1.7b-reward',
]
TARGET_MODEL_DIR = REPO_DIR / 'models'

def find_adapter_dir(adapter_name):
    candidates = [PREFERRED_DRIVE_MODEL_DIR / adapter_name]
    candidates.extend(
        path.parent
        for path in DRIVE_ROOT.rglob('adapter_config.json')
        if path.parent.name == adapter_name
    )
    for candidate in candidates:
        if (candidate / 'adapter_config.json').exists():
            return candidate
    return None

adapter_sources = {}
for name in ADAPTER_NAMES:
    source = find_adapter_dir(name)
    adapter_sources[name] = source
    print(name, '=>', source)

missing = [name for name, source in adapter_sources.items() if source is None]
assert not missing, 'Missing adapter folders in Drive: ' + ', '.join(missing)

TARGET_MODEL_DIR.mkdir(parents=True, exist_ok=True)
for name, source in adapter_sources.items():
    target = TARGET_MODEL_DIR / name
    shutil.rmtree(target, ignore_errors=True)
    shutil.copytree(source, target)
    assert (target / 'adapter_config.json').exists(), f'Copy failed for {name}'
    print('copied', source, '->', target)


## 4. Load Adapters


In [ ]:
import importlib.metadata
import torch

def disable_incompatible_torchao():
    try:
        version = importlib.metadata.version('torchao')
    except importlib.metadata.PackageNotFoundError:
        return
    version_parts = tuple(int(part) for part in version.split('.')[:2] if part.isdigit())
    if version_parts >= (0, 16):
        return
    import peft.import_utils as peft_import_utils
    peft_import_utils.is_torchao_available = lambda: False
    try:
        import peft.tuners.lora.torchao as peft_lora_torchao
        peft_lora_torchao.is_torchao_available = lambda: False
    except Exception:
        pass

disable_incompatible_torchao()
os.chdir(REPO_DIR)
BASE_MODEL = 'Qwen/Qwen3-1.7B'
TEXT_ADAPTER = 'models/rowan-qwen3-1.7b-sft'
REWARD_ADAPTER = 'models/rowan-qwen3-1.7b-reward'

assert Path(TEXT_ADAPTER, 'adapter_config.json').exists(), f'Missing {TEXT_ADAPTER}'
assert Path(REWARD_ADAPTER, 'adapter_config.json').exists(), f'Missing {REWARD_ADAPTER}'
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
print('base model:', BASE_MODEL)
print('text adapter:', TEXT_ADAPTER)
print('reward adapter:', REWARD_ADAPTER)


## 5. Start Interactive Rowan Level


In [ ]:
from game_api import GameNpcEngine

NPC_SYSTEM_PROMPT = """You are Rowan Ashford, the late-twenties owner of The Last Light, a rooftop greenhouse cafe above a shuttered bookshop in Marrow Bay. It is dusk at closing time on the fifth evening the player has appeared at the same corner table by the ferns. The room is warm, overgrown, and unpretentious: jasmine, wet soil, bitter coffee, string lights, old potting benches, rain-warped paperbacks, and Fig the gray cat.

You are precise, dry, quietly observant, guarded, and slow to warm. You notice small specifics and show care through action before words. You trust patterns more than declarations. Reply as Rowan in short, considered sentences. Let silence and subtext matter. Use occasional botanical metaphors naturally. Do not rush romance, confess feelings, become flirtatious too quickly, or explain your inner state directly. If the player is patient, specific, and respectful, allow small openings. If the player pushes, uses generic charm, or treats guardedness like a puzzle to solve, retreat into polite cafe-owner distance."""

REWARD_SYSTEM_PROMPT = """Score higher when the player helps the scene preserve slow-burn intimacy: patience, specificity, attention to accumulated details, respect for Rowan's boundaries, and gentle curiosity. Score lower when the player rushes confession, pushes past deflections, uses generic pickup lines, ignores the setting, demands vulnerability, or breaks character. The ideal trajectory is not immediate romance; it is Rowan making a little more room without naming why."""

engine = GameNpcEngine(
    text_model_path=BASE_MODEL,
    text_adapter_path=TEXT_ADAPTER,
    scorer='trained',
    reward_base_model_path=BASE_MODEL,
    reward_model_path=REWARD_ADAPTER,
)

level = engine.create_level(
    npc_system_prompt=NPC_SYSTEM_PROMPT,
    reward_system_prompt=REWARD_SYSTEM_PROMPT,
    objective='Build quiet trust with Rowan at closing time without rushing the intimacy.',
    pass_score=0.75,
)
session_id = level['session_id']
print('Started Rowan test level.')
print('First turn may be slow while Qwen and the reward model load.')
print('Type exit to stop.')
print('Initial score:', level['score'], '/', level['pass_score'])

while True:
    player = input('You: ').strip()
    if player.lower() in {'exit', 'quit'}:
        break
    turn = engine.send_player_message(
        session_id,
        player,
        max_new_tokens=80,
        temperature=0.7,
        top_p=0.9,
    )
    print('Rowan:', turn['npc_reply'])
    print(f"Score: {turn['score']:.3f} ({turn['score_delta']:+.3f}) passed={turn['passed']}")
    print('Reward:', turn['reward_reason'])
    print('-' * 60)
